**關於這份檔案 / About this file**

這份程式碼負責將原始的問卷資料進行清理與重新編碼，產出全組共用的乾淨資料集，確保所有人的分析基礎 100% 一致。

This script cleans and recodes the raw survey data to generate a unified, clean dataset for the entire team, ensuring our analysis foundation is 100% consistent.

***🗑️ 刪除了什麼資料？ / What data was removed?***

**1.未使用的變數 / Unused variables:**

移除了與本次專題無關的數十個問卷題目，減少干擾。 / Removed dozens of survey questions irrelevant to this project to reduce noise.

**2.缺失值 / Missing values (NaN):**

只要學生在我們選定的三個變數中，有任何一題沒有作答（交白卷），該名學生的整筆資料就會被刪除。 / Any student record containing blank answers in our selected variables was completely dropped.

**3.無效代碼 / Invalid codes:**

在 SadOrHopeless 變數中，移除了不等於 1 或 2 的無效填答。 / Removed invalid responses in the SadOrHopeless variable that were not equal to 1 or 2.

***💾 保留了什麼變數與基準值？ / What variables and baselines were kept?***

經過清理後，產出的 yrbs_cleaned.csv 只包含以下三個用於後續分析的變數：

After cleaning, the generated yrbs_cleaned.csv contains only the following three variables for subsequent analysis:

**1.行為變數 (Behavioral Variable):**

SadOrHopeless_Recoded說明 / Description: 將原本的 SadOrHopeless 重新編碼。1 轉換為 1 (成功/Yes)，2 轉換為 0 (失敗/No)。

/ Recoded from the original SadOrHopeless. 1 is converted to 1 (Success/Yes), and 2 is converted to 0 (Failure/No).

檢定基準值 / Reference value: $p_0 = 0.30$ (供角色一使用 / For Role 1).

**2.連續變數 (Continuous Variable):**

BMIPCT說明 / Description: 學生的 BMI 百分位數，數值保留原樣不變。

 / Students' BMI percentile; values remain unchanged.

檢定基準值 / Reference value: $\mu_0 = 65.0$ (供角色二使用 / For Role 2).

**3.分組變數 (Grouping Variable):**

WhatIsYourSex說明 / Description: 學生的性別，保留供角色三進行額外的進階探索性分析 (Extra EDA) 使用。

 / Students' sex, kept for Role 3 to conduct the extra Exploratory Data Analysis (Extra EDA).

In [21]:
import pandas as pd

# ==========================================
# 1. 讀取原始資料 / Read raw data
# ==========================================
# 依據你的截圖，檔名確認為 YRBS_2007.csv / File name confirmed as YRBS_2007.csv based on your screenshot
file_name = 'YRBS_2007.csv'
print(f"正在讀取 {file_name}...")

try:
    df = pd.read_csv(file_name)
    df.columns = df.columns.str.strip()
    print(f"✅ 讀取成功！原始資料共有 {len(df)} 筆紀錄。")

    # ==========================================
    # 2. 挑選分析變數與處理缺失值 / Select analysis variables and handle missing values
    # ==========================================
    # 修正欄位名稱：將 Sex 改為正確的 WhatIsYourSex / Correct column name: change Sex to the correct WhatIsYourSex
    target_columns = ['SadOrHopeless', 'BMIPCT', 'WhatIsYourSex']

    # 檢查欄位是否存在 / Check if columns exist
    available_cols = [col for col in target_columns if col in df.columns]
    if len(available_cols) < len(target_columns):
        print(f"⚠️ 警告：找不到部分欄位。目前檔案中的欄位有：{df.columns.tolist()[:10]}...")
    else:
        df_selected = df[target_columns].copy()

        # 刪除缺失值 (NaN) / Drop missing values (NaN)
        df_clean = df_selected.dropna().copy()
        print(f"✅ 已刪除缺失值。剩餘紀錄：{len(df_clean)} 筆。")

        # ==========================================
        # 3. 變數過濾與重編碼 / Variable filtering and recoding
        # ==========================================
        df_clean = df_clean[df_clean['SadOrHopeless'].isin([1, 2])].copy()

        # 執行重編碼：1 -> 1 (成功), 2 -> 0 (失敗) / Execute recoding: 1 -> 1 (Success), 2 -> 0 (Failure)
        df_clean['SadOrHopeless_Recoded'] = df_clean['SadOrHopeless'].replace({1: 1, 2: 0})

        print(f"✅ 最終分析樣本數 (n)：{len(df_clean)} 筆。")

        # ==========================================
        # 4. 匯出處理後的資料 / Export processed data
        # ==========================================
        output_name = 'yrbs_cleaned.csv'
        df_clean.to_csv(output_name, index=False)

        print(f"---")
        print(f"🚀 清理完成！請執行以下動作：")
        print(f"1. 下載 '{output_name}'。")
        print(f"2. 上傳至 GitHub 的 'data/processed/' 資料夾。")
        print(f"3. 角色一分析行為變數基準值為 p_0=0.30。")
        # 修正了 \m 的警告問題 / Fixed the \m warning issue
        print(f"4. 角色二分析連續變數基準值為 \\mu_0=65.0。")

except FileNotFoundError:
    print(f"❌ 找不到檔案！請確認左側檔名是否為 '{file_name}'。")

正在讀取 YRBS_2007.csv...
✅ 讀取成功！原始資料共有 14041 筆紀錄。
✅ 已刪除缺失值。剩餘紀錄：12902 筆。
✅ 最終分析樣本數 (n)：12902 筆。
---
🚀 清理完成！請執行以下動作：
1. 下載 'yrbs_cleaned.csv'。
2. 上傳至 GitHub 的 'data/processed/' 資料夾。
3. 角色一分析行為變數基準值為 p_0=0.30。
4. 角色二分析連續變數基準值為 \mu_0=65.0。
